# 05 · Machine Learning con MLlib

**Objetivos**
- Preparar características con `StringIndexer`, `OneHotEncoder` y `VectorAssembler` (Meng et al., 2016).
- Construir y evaluar un `Pipeline` de **regresión**.
- Segmentar clientes con **K-Means** (variables RFM).

> Duración estimada: 30 min

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Fuera de Docker -> local[*]. Dentro del docker-compose -> spark://spark-master:7077
MASTER = os.environ.get("SPARK_MASTER", "local[*]")

builder = (SparkSession.builder
           .appName("05_MLlib")
           .master(MASTER)
           .config("spark.sql.shuffle.partitions", "8"))   # pocos datos -> pocas particiones
if os.environ.get("SPARK_DRIVER_HOST"):
    builder = builder.config("spark.driver.host", os.environ["SPARK_DRIVER_HOST"])

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

DATA = os.path.abspath("../data")   # carpeta de datos del repositorio
print("Spark", spark.version, "| master:", sc.master)
print("Spark UI:", sc.uiWebUrl)

In [ ]:
ventas = (spark.read.csv(f"{DATA}/ventas.csv", header=True, inferSchema=True)
          .filter(F.col("cantidad") > 0)
          .fillna({"canal": "Desconocido"})
          .withColumn("total", F.col("cantidad") * F.col("precio_unitario") * (1 - F.col("descuento"))))
productos = spark.read.json(f"{DATA}/productos.json")
clientes = spark.read.csv(f"{DATA}/clientes.csv", header=True, inferSchema=True)

df = (ventas.join(productos.select("id_producto", "categoria"), "id_producto")
            .join(clientes.select("id_cliente", "edad", "segmento", "region"), "id_cliente"))
df.select("cantidad", "precio_unitario", "descuento", "categoria", "canal", "edad", "total").show(5)

## Parte A · Regresión: predicción del valor total de una venta

Flujo típico en MLlib (The Apache Software Foundation [ASF], 2026c):

```
columnas texto ─► StringIndexer ─► OneHotEncoder ─┐
                                                   ├─► VectorAssembler ─► Modelo
columnas numéricas ────────────────────────────────┘
```

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

categoricas = ["categoria", "canal", "segmento", "region"]
numericas   = ["cantidad", "precio_unitario", "descuento", "edad"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categoricas]
encoder  = OneHotEncoder(inputCols=[f"{c}_idx" for c in categoricas],
                         outputCols=[f"{c}_ohe" for c in categoricas])
assembler = VectorAssembler(inputCols=numericas + [f"{c}_ohe" for c in categoricas],
                            outputCol="features")
rf = RandomForestRegressor(featuresCol="features", labelCol="total", numTrees=40, maxDepth=8, seed=42)

pipeline = Pipeline(stages=indexers + [encoder, assembler, rf])

In [ ]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
print(f"Entrenamiento: {train.count():,} | Prueba: {test.count():,}")

modelo = pipeline.fit(train)
pred = modelo.transform(test)
pred.select("categoria", "cantidad", "precio_unitario", "descuento", "total", "prediction").show(5)

In [ ]:
for metrica in ["rmse", "mae", "r2"]:
    ev = RegressionEvaluator(labelCol="total", predictionCol="prediction", metricName=metrica)
    print(f"{metrica.upper():>4}: {ev.evaluate(pred):,.3f}")

### Importancia de las variables

In [ ]:
import pandas as pd

attrs = pred.schema["features"].metadata["ml_attr"]["attrs"]
nombres = sorted([a for grupo in attrs.values() for a in grupo], key=lambda a: a["idx"])
importancias = modelo.stages[-1].featureImportances.toArray()

(pd.DataFrame({"variable": [a["name"] for a in nombres], "importancia": importancias})
   .sort_values("importancia", ascending=False).head(8))

> Pregunta de análisis: ¿es coherente que `precio_unitario` y `cantidad` sean las variables más importantes? ¿Por qué `edad` apenas aporta al modelo?

## Parte B · Segmentación de clientes con K-Means (RFM)

- **R**ecencia: días desde la última compra
- **F**recuencia: número de compras
- **M**onto: total gastado

In [ ]:
rfm = (ventas.groupBy("id_cliente")
       .agg(F.datediff(F.lit("2026-01-01"), F.max("fecha")).alias("recencia"),
            F.count("*").alias("frecuencia"),
            F.sum("total").alias("monto")))
rfm.describe().show()

In [ ]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

vec = VectorAssembler(inputCols=["recencia", "frecuencia", "monto"], outputCol="rfm_vec").transform(rfm)
scaler = StandardScaler(inputCol="rfm_vec", outputCol="features", withMean=True, withStd=True).fit(vec)
rfm_esc = scaler.transform(vec)

# Elegir k con el coeficiente de silueta
evaluador = ClusteringEvaluator()
for k in range(2, 7):
    m = KMeans(k=k, seed=42).fit(rfm_esc)
    print(f"k={k}  silueta={evaluador.evaluate(m.transform(rfm_esc)):.3f}")

In [ ]:
kmeans = KMeans(k=4, seed=42).fit(rfm_esc)
segmentos = kmeans.transform(rfm_esc)

(segmentos.groupBy("prediction")
 .agg(F.count("*").alias("clientes"),
      F.round(F.avg("recencia")).alias("recencia_prom"),
      F.round(F.avg("frecuencia"), 1).alias("frecuencia_prom"),
      F.round(F.avg("monto") / 1e6, 2).alias("monto_prom_mill"))
 .orderBy("prediction")
 .show())

In [ ]:
import matplotlib.pyplot as plt

pdf = segmentos.select("frecuencia", "monto", "prediction").toPandas()
plt.figure(figsize=(7, 5))
plt.scatter(pdf["frecuencia"], pdf["monto"] / 1e6, c=pdf["prediction"], cmap="viridis", s=18)
plt.xlabel("Frecuencia (n° compras)"); plt.ylabel("Monto (millones COP)")
plt.title("Segmentos de clientes (K-Means sobre RFM)")
plt.colorbar(label="segmento")
plt.show()

## Guardar y cargar el modelo

In [ ]:
from pyspark.ml import PipelineModel

ruta = os.path.abspath("../salida/modelo_rf")
modelo.write().overwrite().save(ruta)
cargado = PipelineModel.load(ruta)
cargado.transform(test).select("total", "prediction").show(3)

## Ejercicios propuestos

1. Reemplazar `RandomForestRegressor` por `LinearRegression` y comparar el R².
2. Emplear `CrossValidator` y `ParamGridBuilder` para probar `numTrees` ∈ {20, 50} y `maxDepth` ∈ {5, 10}.
3. Asignar un **nombre de negocio** a cada segmento de clientes (por ejemplo, *Leales* o *En riesgo*) con base en sus valores promedio.

In [ ]:
# Ejercicio 1

In [ ]:
spark.stop()

## Referencias

Meng, X., Bradley, J., Yavuz, B., Sparks, E., Venkataraman, S., Liu, D., Freeman, J., Tsai, D. B., Amde, M., Owen, S., Xin, D., Xin, R., Franklin, M. J., Zadeh, R., Zaharia, M., & Talwalkar, A. (2016). MLlib: Machine learning in Apache Spark. *Journal of Machine Learning Research, 17*(34), 1–7. https://jmlr.org/papers/v17/15-237.html

The Apache Software Foundation. (2026c). *Machine learning library (MLlib) guide* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/ml-guide.html